# WisdomLens Data Analysis Baseline

This notebook performs a read-only exploratory analysis of the WisdomLens dataset from PostgreSQL.

Objectives:
- Inspect the actual schema currently used by the app
- Build a reproducible analysis baseline using Python and Pandas
- Explore questions, answers, perspectives, and timing patterns
- Report data quality issues without modifying production data

## 1. Setup

This notebook reuses the projects existing backend configuration and database connection instead of hardcoding credentials.

In [ ]:
import json
import locale
import os
import re
import string
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"

try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

try:
    locale.setlocale(locale.LC_ALL, "C.UTF-8")
except Exception:
    pass

current_dir = Path.cwd().resolve()
repo_root = None
backend_root = None

for candidate in [current_dir, *current_dir.parents]:
    candidate_backend = candidate / 'backend'
    if candidate_backend.exists():
        repo_root = candidate
        backend_root = candidate_backend
        break

if repo_root is None or backend_root is None:
    raise FileNotFoundError('Could not find backend folder from the current working directory.')

if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

from sqlalchemy import inspect, select

from app.config import get_database_url
from app.database import SessionLocal
from app.models.inquiry import Inquiry


def safe_df_as_string(df, rows=5):
    preview = df.head(rows).copy()
    return json.dumps(preview.to_dict(orient='records'), ensure_ascii=True, indent=2, default=str)


print(f'Current working directory: {current_dir}')
print(f'Repo root: {repo_root}')
print(f'Backend root: {backend_root}')
print(f'Database URL loaded: {get_database_url()}')


## 2. Inspect the actual schema

The project currently uses `inquiries` as the main stored user interaction table. This notebook checks the real schema before doing any analysis.

In [ ]:
with SessionLocal() as db:
    inspector = inspect(db.bind)
    table_names = inspector.get_table_names()
    print('Tables found:', table_names)

    if 'inquiries' in table_names:
        columns = inspector.get_columns('inquiries')
        print('\nColumns in inquiries:')
        for col in columns:
            print(f" - {col['name']}: {col['type']}")
    else:
        print('The inquiries table is not available in the current database.')

## 3. Load the primary dataset from PostgreSQL

This notebook uses `inquiries` as the primary dataset, because this project stores question/answer history there. The exact fields are read from the live schema.

In [ ]:
with SessionLocal() as db:
    result = db.execute(
        select(
            Inquiry.id,
            Inquiry.question,
            Inquiry.summary,
            Inquiry.perspectives,
            Inquiry.buddhism,
            Inquiry.western_philosophy,
            Inquiry.psychology,
            Inquiry.language,
            Inquiry.source,
            Inquiry.model,
            Inquiry.created_at,
        )
    ).all()

columns = [
    'id', 'question', 'summary', 'perspectives',
    'buddhism', 'western_philosophy', 'psychology',
    'language', 'source', 'model', 'created_at'
]

df = pd.DataFrame(result, columns=columns)
print(safe_df_as_string(df, 5))

## 4. Dataset overview

This step shows dataset size and basic column-level statistics.

In [ ]:
print('Rows:', len(df))
print('Columns:', len(df.columns))
df.info()

In [ ]:
overview = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[col].dtype) for col in df.columns],
    'non_null': [int(df[col].notna().sum()) for col in df.columns],
    'null_count': [int(df[col].isna().sum()) for col in df.columns],
    'null_pct': [round(df[col].isna().mean() * 100, 2) for col in df.columns],
})

overview

## 5. Duplicate analysis

This section reports fully duplicated rows and potentially duplicated question text without deleting anything.

In [ ]:
# Use a hashable representation because some columns contain JSONB/dict values.
duplicate_rows = int(
    df.apply(
        lambda row: tuple(
            json.dumps(value, ensure_ascii=False, default=str)
            if isinstance(value, (dict, list))
            else value
            for value in row
        ),
        axis=1,
    ).duplicated(keep=False).sum()
)
print(f'Fully duplicated rows: {duplicate_rows}')

question_duplicates = df[df['question'].notna()].duplicated(subset=['question'], keep=False)
print(f'Potentially duplicated questions: {int(question_duplicates.sum())}')

In [ ]:
question_duplicate_rows = df[df['question'].notna()].loc[
    lambda x: x.duplicated(subset=['question'], keep=False)
].head(20)
print(json.dumps(question_duplicate_rows.to_dict(orient='records'), ensure_ascii=True, indent=2, default=str))

## 6. Question analysis

Analyze question length using both character count and word count.

In [ ]:
df['question_len_chars'] = df['question'].fillna('').str.len()
df['question_len_words'] = df['question'].fillna('').str.split().str.len()

print('Question length (characters):')
print(df['question_len_chars'].describe())
print('\nQuestion length (words):')
print(df['question_len_words'].describe())

In [ ]:
short_questions = df[df['question_len_chars'] <= 20][['id', 'question', 'question_len_chars']].head(20)
long_questions = df[df['question_len_chars'] >= df['question_len_chars'].quantile(0.95)][['id', 'question', 'question_len_chars']].head(20)

print('Very short questions:')
print(json.dumps(short_questions.to_dict(orient='records'), ensure_ascii=True, indent=2, default=str))

print('Very long questions:')
print(json.dumps(long_questions.to_dict(orient='records'), ensure_ascii=True, indent=2, default=str))

## 7. Answer analysis

In this project, answer data is stored in the `summary` field and in `perspectives` JSONB. This step analyzes both.

In [ ]:
df['summary_len_chars'] = df['summary'].fillna('').str.len()
df['summary_len_words'] = df['summary'].fillna('').str.split().str.len()

print('Summary length (characters):')
print(df['summary_len_chars'].describe())
print('\nSummary length (words):')
print(df['summary_len_words'].describe())

In [ ]:
for field in ['buddhism', 'western_philosophy', 'psychology']:
    if field in df.columns:
        df[f'{field}_len_chars'] = df[field].fillna('').str.len()
        df[f'{field}_len_words'] = df[field].fillna('').str.split().str.len()

## 8. Perspective distribution

Do not assume only three perspectives exist. Read the keys from the actual `perspectives` JSONB and report the distribution.

In [ ]:
def normalize_perspective_dict(value):
    if pd.isna(value):
        return {}
    try:
        if isinstance(value, str):
            value = json.loads(value)
        if isinstance(value, dict):
            return {str(k): str(v) for k, v in value.items() if v is not None}
        return {}
    except Exception:
        return {}

df['perspectives_dict'] = df['perspectives'].map(normalize_perspective_dict)

all_perspective_keys = []
for item in df['perspectives_dict']:
    all_perspective_keys.extend(item.keys())

perspective_counts = pd.Series(all_perspective_keys).value_counts()
print('Perspective counts from perspectives JSONB:')
print(perspective_counts)

In [ ]:
if not perspective_counts.empty:
    plt.figure(figsize=(10, 5))
    perspective_counts.plot(kind='bar', color='steelblue')
    plt.title('Perspective distribution')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 9. Time-based analysis

If `created_at` exists, compute counts per day, week, and month.

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')

daily_counts = df.groupby(df['created_at'].dt.floor('D')).size().rename('count')
weekly_counts = df.groupby(df['created_at'].dt.floor('W')).size().rename('count')
monthly_counts = df.groupby(df['created_at'].dt.to_period('M')).size().rename('count')

print('Daily counts:')
print(daily_counts.head(10).to_string())

print('Weekly counts:')
print(weekly_counts.head(10).to_string())

print('Monthly counts:')
print(monthly_counts.head(10).to_string())

In [ ]:
if df['created_at'].notna().any():
    plt.figure(figsize=(12, 5))
    daily_counts.plot()
    plt.title('Questions per day')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 10. Basic text exploration

Perform a lightweight text analysis of the question field, keeping the implementation simple.

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[0-9]+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['question_clean'] = df['question'].map(clean_text)

all_words = ' '.join(df['question_clean']).split()
word_counts = pd.Series(all_words).value_counts().head(20)

print('Top words in questions:')
print(word_counts)

In [ ]:
stopwords = {
    'và', 'của', 'là', 'các', 'một', 'nếu', 'về', 'để', 'như', 'the', 'a', 'an', 'to', 'of', 'in', 'for', 'on', 'with', 'why', 'how'
}

filtered_words = [w for w in all_words if w not in stopwords]
filtered_counts = pd.Series(filtered_words).value_counts().head(20)

print('Top meaningful words after basic stopword-style filtering:')
print(filtered_counts)

## 11. Perspective × question statistics

If the data supports it, compare perspectives using question length and answer length metrics.

In [ ]:
perspective_rows = []

for _, row in df.iterrows():
    for perspective_name in row['perspectives_dict'].keys():
        perspective_rows.append({
            'id': row['id'],
            'question': row['question'],
            'summary': row['summary'],
            'perspective': perspective_name,
            'question_len_chars': row['question_len_chars'],
            'summary_len_chars': row['summary_len_chars'],
        })

perspective_df = pd.DataFrame(perspective_rows)

perspective_summary = perspective_df.groupby('perspective').agg(
    questions_count=('id', 'count'),
    avg_question_length=('question_len_chars', 'mean'),
    median_question_length=('question_len_chars', 'median'),
    avg_answer_length=('summary_len_chars', 'mean'),
    median_answer_length=('summary_len_chars', 'median'),
)

print(perspective_summary.to_string())

## 12. Data quality summary

This section summarizes missing values, duplicates, available perspectives, and date coverage.

In [ ]:
missing_report = pd.DataFrame({
    'column': df.columns,
    'null_count': [int(df[col].isna().sum()) for col in df.columns],
    'null_pct': [round(df[col].isna().mean() * 100, 2) for col in df.columns],
}).sort_values('null_pct', ascending=False)

print('Missing data report:')
print(missing_report.to_string())

In [ ]:
data_quality = {
    'dataset_size': int(len(df)),
    'duplicate_rows': int(
        df.apply(
            lambda row: tuple(
                json.dumps(value, ensure_ascii=False, default=str)
                if isinstance(value, (dict, list))
                else value
                for value in row
            ),
            axis=1,
        ).duplicated(keep=False).sum()
    ),
    'duplicate_question_rows': int(df[df['question'].notna()].duplicated(subset=['question'], keep=False).sum()),
    'available_perspectives': sorted(set(all_perspective_keys)),
    'date_coverage_start': str(df['created_at'].min()),
    'date_coverage_end': str(df['created_at'].max()),
    'potential_issues': []
}

if df['question'].isna().any():
    data_quality['potential_issues'].append('Some rows have missing question text.')
if df['summary'].isna().any():
    data_quality['potential_issues'].append('Some rows have missing summary text.')
if df['created_at'].isna().any():
    data_quality['potential_issues'].append('Some rows have missing created_at values.')

print(json.dumps(data_quality, indent=2, default=str))

## 13. Initial observations

This section should only summarize the results directly supported by the analysis above.

In [ ]:
initial_observations = {
    'records_total': int(len(df)),
    'records_with_question': int(df['question'].notna().sum()),
    'records_with_summary': int(df['summary'].notna().sum()),
    'available_perspectives': sorted(set(all_perspective_keys)),
    'duplicate_questions': int(df[df['question'].notna()].duplicated(subset=['question'], keep=False).sum()),
    'date_range_start': str(df['created_at'].min()),
    'date_range_end': str(df['created_at'].max()),
}

print(json.dumps(initial_observations, indent=2, default=str))

## 14. Next steps

Suggested Phase 2 directions (not implemented in this notebook):
- Topic extraction
- Semantic clustering
- Question similarity
- Question × perspective relationships
- AI-assisted pattern discovery
- Hypothesis generation
- Statistical validation of AI-generated hypotheses